# Whisper 기반 한국어 명령어 분류

원본의 핵심 코드를 정리한 실행 흐름입니다. 11개 명령어를 분류하고 학습 특징의 중심과 먼 입력을 Unknown(-1)으로 거절합니다.

이 정리본은 재학습 결과를 포함하지 않습니다. 원래 저장 결과와 자료별 차이는 `docs/RESULTS.md`, 수정 범위는 `docs/REPRODUCIBILITY.md`를 참고하세요. 저장소 전체, requirements의 패키지, FFmpeg, 본인의 음성 자료가 필요합니다. Colab에서는 저장소 전체를 준비하고 데이터 경로를 자신의 Drive 경로로 수정하세요.

In [ ]:
from pathlib import Path
import os, sys, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("저장소 또는 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT))
from src.config import ORDERED_LABELS, NUM_LABELS, TARGET_SAMPLES
from src.data import RawAudioDataset, create_metadata
from src.model import IntegratedWhisperFFT
from src.unknown import extract_embeddings_from_loader, build_prototypes, ModelWithUnknownRaw
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
print("device:", DEVICE)

## 데이터 준비

`KWS_DATA_DIR` 또는 `BASE_PATH`를 지정하세요. 두 폴더의 WAV 파일은 `1-1_speaker01_take01.wav`처럼 명령어 코드로 시작해야 합니다. 화자가 Train과 Test에 겹치지 않는지는 직접 확인해야 합니다. 로더는 명령어만 읽으며 화자 분할을 자동 수행하지 않습니다.

In [ ]:
BASE_PATH = Path(os.environ.get("KWS_DATA_DIR", str(ROOT / "data"))).expanduser()
train_df = create_metadata(BASE_PATH / "Train Dataset")
test_df = create_metadata(BASE_PATH / "Test Dataset")
if train_df.empty or test_df.empty:
    raise ValueError("Train/Test 폴더에 명령어 코드로 시작하는 WAV 파일을 준비하세요.")
if set(train_df["label"]) != set(range(NUM_LABELS)):
    raise ValueError("학습 폴더에 11개 명령어가 모두 있어야 합니다.")
print("Train:", len(train_df), "Test:", len(test_df))
train_dataset = RawAudioDataset(train_df, TARGET_SAMPLES)
test_dataset = RawAudioDataset(test_df, TARGET_SAMPLES)
# Windows/Jupyter에서 프로세스 생성 의존성을 줄이는 실행 설정
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

## 학습

Whisper small 인코더 전체를 미세조정합니다. 8 epoch와 매 epoch의 Test 모니터링은 원본 절차입니다. 새로운 비교 실험은 화자별 Train/Validation/Test를 분리하고 Test를 설정 선택에 사용하지 않아야 합니다.

In [ ]:
# ---------------------------------------------------------
# 5. 학습 루프 (raw audio)
# ---------------------------------------------------------
model = IntegratedWhisperFFT(NUM_LABELS, whisper_size="small").to(DEVICE)
print("✅ Raw->Mel->WhisperEncoder 모델 생성 완료")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)
EPOCHS = 8

train_losses = []
test_accs = []

print(f"\n🚀 학습 시작 (총 {EPOCHS} Epochs)")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for audio, labels in pbar:
        audio, labels = audio.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(audio)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': float(loss.item())})

    avg_loss = running_loss / max(1, len(train_loader))
    train_losses.append(avg_loss)

    # Test
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for audio, labels in test_loader:
            audio, labels = audio.to(DEVICE), labels.to(DEVICE)
            logits = model(audio)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total if total > 0 else 0
    test_accs.append(acc)
    print(f"   [Epoch {epoch+1}] Loss: {avg_loss:.4f} | Test Acc: {acc*100:.2f}%")




(OUT / "training_history.json").write_text(json.dumps({"train_loss":train_losses,"test_accuracy":test_accs},indent=2),encoding="utf-8")

## 알려진 명령어 평가

이 정확도는 등록된 11개 명령어의 분류 성능입니다. 등록하지 않은 단어를 거절하는 능력과는 다릅니다.

In [ ]:
model.eval()
all_true, all_pred = [], []
with torch.no_grad():
    for audio, labels in test_loader:
        pred = model(audio.to(DEVICE)).argmax(dim=1).cpu()
        all_true.extend(labels.tolist())
        all_pred.extend(pred.tolist())
print(classification_report(all_true, all_pred, labels=list(range(NUM_LABELS)), target_names=ORDERED_LABELS, zero_division=0))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
epochs = range(1, len(train_losses)+1)
axes[0].plot(epochs, train_losses, marker="o")
axes[0].set(xlabel="Epoch", ylabel="Train loss")
axes[1].plot(epochs, test_accs, marker="o")
axes[1].set(xlabel="Epoch", ylabel="Known-command test accuracy", ylim=(0, 1.05))
fig.tight_layout()
plt.show()

## 클래스 중심과 거리 기준

정리본은 학습과 거리분석에 동일한 RawAudio 전처리를 사용합니다. 원본 후반의 무음 제거+별도 Mel 경로와 달라 결과가 달라질 수 있습니다. 학습 거리 p99는 정답 확률 99%가 아닙니다. 별도 Unknown 검증 자료로 기준을 선택하는 것은 후속 과제입니다.

In [ ]:
train_E, train_y = extract_embeddings_from_loader(model, train_loader, DEVICE)
prototypes = build_prototypes(train_E, train_y, NUM_LABELS)
dist_all = 1.0 - train_E @ prototypes.T
train_dtrue = dist_all[torch.arange(len(train_y)), train_y]
percentiles = [90, 95, 97, 99]
candidates = np.percentile(train_dtrue.numpy(), percentiles)
print(dict(zip(percentiles, candidates.tolist())))
# 현재 학습 자료의 p99. 원본 export 코드는 고정 0.125 사용.
threshold = float(candidates[-1])
wrapped = ModelWithUnknownRaw(model, prototypes, threshold=threshold).to(DEVICE).eval()
print("Unknown if nearest cosine distance >", threshold)

## 거절 적용 후 평가

Test에 실제 Unknown 정답 자료가 없다면 Unknown 탐지율을 측정할 수 없습니다. 여기서는 알려진 명령어를 얼마나 거절하는지 확인합니다. `conf`는 Softmax 최댓값이며 보정된 정답 확률이 아닙니다.

In [ ]:
rejected, correct, total = 0, 0, 0
with torch.no_grad():
    for audio, labels in test_loader:
        pred, conf, distance = wrapped(audio.to(DEVICE))
        pred = pred.cpu()
        rejected += int((pred == -1).sum())
        correct += int((pred == labels).sum())
        total += len(labels)
summary = {"known_test_samples":total,"correct":correct,"rejected_known":rejected,
           "known_accuracy_with_rejection":correct/total,"threshold":threshold}
print(summary)
(OUT / "unknown_evaluation.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")

## 저장

학습 가중치·프로토타입·임계값은 로컬 파일로 저장하며 Git에서 제외합니다.

In [ ]:
torch.save({"state_dict":model.state_dict(), "prototypes":prototypes,
    "threshold":threshold, "labels":ORDERED_LABELS,
    "whisper_size":"small", "target_samples":TARGET_SAMPLES,
}, OUT / "whisper_commands.pt")
print("Saved:", OUT / "whisper_commands.pt")

## 선택: 과거 모바일 내보내기 방식

원본의 TorchScript Lite 내보내기는 사용하는 PyTorch와 기기의 지원 여부를 별도 확인해야 합니다. 원본 저장 파일은 약 1.03 GB였습니다. 아래는 기본 비활성화 상태이며, 실행하더라도 Android의 연산 지원·지연시간·메모리가 검증되는 것은 아닙니다. 앱 소스는 포함되지 않았습니다.

In [ ]:
EXPORT_LEGACY_LITE = False
if EXPORT_LEGACY_LITE:
    base_cpu = model.to("cpu").eval()
    wrapped_cpu = ModelWithUnknownRaw(base_cpu, prototypes.cpu(), threshold).eval()
    example_audio = torch.zeros(1, TARGET_SAMPLES, dtype=torch.float32)
    with torch.no_grad():
        traced = torch.jit.trace(wrapped_cpu, example_audio, strict=False)
        traced = torch.jit.freeze(traced)
        traced_opt = torch.jit.optimize_for_inference(traced)
        traced_opt._save_for_lite_interpreter(str(OUT / "whisper_kws_mobile.ptl"))
    print("Exported legacy Lite model; device compatibility is not verified.")